# EasyChair Smart CFP — подготовка данных для ЛР-6

Последовательно соберём таблицы **всех Areas**, объединим записи через pandas,
сохраним конференции и их тематические принадлежности, затем выгрузим XLSX.
Источник: https://easychair.org/cfp/area. Это снимок активных CFP, доступных в Areas
на дату скачивания. Поля берём из таблиц: названия, место, дедлайн, дата начала и темы.

Запуск: `uv sync`, затем `uv run jupyter lab easychair_cfp.ipynb` из каталога проекта.
В VS Code выбрать ядро `.venv/Scripts/python.exe`. Выполнять ячейки сверху вниз.

In [1]:
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlsplit
import math
import re
import textwrap

import pandas as pd
import pycountry
from IPython.display import display

from easychair import AREA_INDEX_URL, discover_areas, fetch_html, make_session, parse_cfps

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 80)

## 1. Настройки и кэш

`REFRESH=False` использует уже скачанные страницы и позволяет продолжить после обрыва.
Чтобы получить новый снимок сайта, установить `REFRESH=True` и повторить сбор целиком.
У кэшированных страниц сохраняется исходная дата скачивания, а не дата повторного запуска.
Запросы идут последовательно, с паузой; HTTP-ошибки повторяются ограниченное число раз.

In [2]:
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "pyproject.toml").exists():
    raise RuntimeError("Запустите notebook из каталога EasyChair-Smart-CFP")

CACHE_DIR = PROJECT_DIR / "data" / "cache"
OUTPUT_DIR = PROJECT_DIR / "data" / "processed"
OUTPUT_PATH = OUTPUT_DIR / "easychair_cfp.xlsx"
REFRESH = False
REQUEST_DELAY = 1.0
RUN_AT_UTC = datetime.now(timezone.utc).isoformat(timespec="seconds")
session = make_session()

## 2. Находим все Areas

Не задаём список номеров вручную: читаем ссылки на странице By area.
Количество разделов и их названия ниже позволяют проверить охват сбора.

In [3]:
index_html, index_meta = fetch_html(
    session, AREA_INDEX_URL, CACHE_DIR, "area_index", refresh=REFRESH, delay=REQUEST_DELAY,
)
areas_df = discover_areas(index_html)
print(f"Найдено Areas: {len(areas_df)}; индекс скачан {index_meta['fetched_at_utc']}")
display(areas_df)

Найдено Areas: 21; индекс скачан 2026-10-08T21:13:12+00:00


,area_id,area,area_url
0,6,Accounting and Finance,https://easychair.org/cfp/area?area=6
1,8,Art and Humanities,https://easychair.org/cfp/area?area=8
2,11,Biological Sciences,https://easychair.org/cfp/area?area=11
3,5,Business and Management,https://easychair.org/cfp/area?area=5
4,3,Chemistry,https://easychair.org/cfp/area?area=3
5,23,Civil Engineering,https://easychair.org/cfp/area?area=23
6,1,Computing,https://easychair.org/cfp/area?area=1
7,12,Earth and Environmental Sciences,https://easychair.org/cfp/area?area=12
8,15,Economics,https://easychair.org/cfp/area?area=15
9,16,Education Science,https://easychair.org/cfp/area?area=16


## 3. Собираем таблицы и проверяем полноту

Текущий HTML содержит все строки таблицы. Парсер сверяет их число с показателем
`entries of` на странице; если сайт изменит пагинацию, сбор остановится с объяснением.
У маленьких таблиц сайт отключает навигацию и не показывает отдельный счётчик:
для них фиксируем `count_check=navigation_disabled`, а reported_count оставляем пустым.
Пустой Area допустим. Ошибка запроса или разбора не превращается в пустой раздел.
После ошибки повторить эту ячейку: успешно скачанные страницы уже находятся в кэше.

In [4]:
frames, audit_rows = [], []
for area in areas_df.to_dict("records"):
    audit = {**area, "reported_count": pd.NA, "parsed_count": pd.NA,
             "fetched_at_utc": pd.NA, "from_cache": pd.NA, "count_check": pd.NA, "error": pd.NA}
    try:
        html, meta = fetch_html(
            session, area["area_url"], CACHE_DIR, f"area_{area['area_id']}",
            refresh=REFRESH, delay=REQUEST_DELAY,
        )
        frame, reported = parse_cfps(html, area, meta["fetched_at_utc"])
        frames.append(frame)
        audit.update(reported_count=reported, parsed_count=len(frame),
                     fetched_at_utc=meta["fetched_at_utc"], from_cache=meta["from_cache"],
                     count_check="site_count" if reported is not None else "navigation_disabled")
        print(f"{area['area']}: {len(frame)} строк" + (" [кэш]" if meta["from_cache"] else ""))
    except Exception as error:
        audit["error"] = f"{type(error).__name__}: {error}"
        print(f"{area['area']}: ОШИБКА — {error}")
    audit_rows.append(audit)

audit_df = pd.DataFrame(audit_rows)
display(audit_df)
if audit_df["error"].notna().any():
    raise RuntimeError("Не все Areas собраны. Исправьте ошибки и повторите сбор; XLSX пока не экспортируется.")
raw_records_df = pd.concat(frames, ignore_index=True)
if raw_records_df.empty:
    raise RuntimeError("Во всех Areas нет записей; проверьте источник")
assert len(audit_df) == len(areas_df)
has_count = audit_df["reported_count"].notna()
assert audit_df.loc[has_count, "reported_count"].eq(audit_df.loc[has_count, "parsed_count"]).all()
print(f"Всего записей с повторениями между Areas: {len(raw_records_df)}")
display(raw_records_df.head())

Accounting and Finance: 10 строк [кэш]
Art and Humanities: 85 строк [кэш]
Biological Sciences: 24 строк [кэш]
Business and Management: 88 строк [кэш]
Chemistry: 8 строк [кэш]
Civil Engineering: 35 строк [кэш]
Computing: 609 строк [кэш]


Earth and Environmental Sciences: 43 строк [кэш]
Economics: 35 строк [кэш]
Education Science: 94 строк [кэш]
Energy: 62 строк [кэш]
Engineering: 464 строк [кэш]
Genomics and Bioinformatics: 13 строк [кэш]
Health Sciences: 43 строк [кэш]
Language and Linguistics: 48 строк [кэш]
Materials Science: 18 строк [кэш]
Mathematics and Statistics: 68 строк [кэш]


Mechanical Engineering: 30 строк [кэш]
Physics: 16 строк [кэш]
Social Sciences: 119 строк [кэш]
Technology: 526 строк [кэш]


,area_id,area,area_url,reported_count,parsed_count,fetched_at_utc,from_cache,count_check,error
0,6,Accounting and Finance,https://easychair.org/cfp/area?area=6,10.0,10,2026-10-08T21:13:13+00:00,True,site_count,<NA>
1,8,Art and Humanities,https://easychair.org/cfp/area?area=8,85.0,85,2026-10-08T21:13:15+00:00,True,site_count,<NA>
2,11,Biological Sciences,https://easychair.org/cfp/area?area=11,24.0,24,2026-10-08T21:13:16+00:00,True,site_count,<NA>
3,5,Business and Management,https://easychair.org/cfp/area?area=5,88.0,88,2026-10-08T21:13:18+00:00,True,site_count,<NA>
4,3,Chemistry,https://easychair.org/cfp/area?area=3,NaN,8,2026-10-08T21:13:19+00:00,True,navigation_disabled,<NA>
5,23,Civil Engineering,https://easychair.org/cfp/area?area=23,35.0,35,2026-10-08T21:13:20+00:00,True,site_count,<NA>
6,1,Computing,https://easychair.org/cfp/area?area=1,609.0,609,2026-10-08T21:13:22+00:00,True,site_count,<NA>
7,12,Earth and Environmental Sciences,https://easychair.org/cfp/area?area=12,43.0,43,2026-10-08T21:13:24+00:00,True,site_count,<NA>
8,15,Economics,https://easychair.org/cfp/area?area=15,35.0,35,2026-10-08T21:13:25+00:00,True,site_count,<NA>
9,16,Education Science,https://easychair.org/cfp/area?area=16,94.0,94,2026-10-08T21:13:26+00:00,True,site_count,<NA>


Всего записей с повторениями между Areas: 2438


,cfp_url,acronym,name,location_raw,submission_deadline_raw,start_date_raw,topics,area_id,area,area_url,fetched_at_utc
0,https://easychair.org/cfp/SAMIBC2027,SAMIBC2027,SAM International Business Conference,"Jacksonville, Florida, United States","March 1, 2027","March 15, 2027",[],6,Accounting and Finance,https://easychair.org/cfp/area?area=6,2026-10-08T21:13:13+00:00
1,https://easychair.org/cfp/2ICAH,2ICAH,II International Congress of Accounting History,"Porto, Portugal","March 31, 2027","October 14, 2027","[history, accounting, auditing, accountability]",6,Accounting and Finance,https://easychair.org/cfp/area?area=6,2026-10-08T21:13:13+00:00
2,https://easychair.org/cfp/qai4fin,QAI4Fin 2026,Quantum-Enhanced AI for Financial Decision-Making,"Milan, Italy","September 30, 2026","November 14, 2026","[artificial intelligence, quantum computing, finance]",6,Accounting and Finance,https://easychair.org/cfp/area?area=6,2026-10-08T21:13:13+00:00
3,https://easychair.org/cfp/ecaes2026,ECAES 2026,VII Congreso Internacional - Economía y Contabilidad Aplicado a la Empresa y...,"Ambato, Ecuador","September 16, 2026","September 23, 2026",[economía],6,Accounting and Finance,https://easychair.org/cfp/area?area=6,2026-10-08T21:13:13+00:00
4,https://easychair.org/cfp/incosef2026,INCOSEF 2026,INCOSEF EUMMAS A2S 2026,"Ho Chi Minh City, Viet Nam","December 8, 2026","December 10, 2026","[global macroeconomic uncertainty, sustainable finance esg climate risk, fin...",6,Accounting and Finance,https://easychair.org/cfp/area?area=6,2026-10-08T21:13:13+00:00


## 4. Уникальные конференции и связи с Areas

Идентификатор — последний сегмент ссылки CFP, а не название конференции.
Одна конференция может принадлежать нескольким Areas: сохраняем все пары в
`conference_areas_df`, а в `conferences_df` оставляем одну строку на CFP.

При разных значениях одного поля в разных Areas сохраняется первое непустое значение
в порядке названий Areas. Такие расхождения попадают в `quality_issues`, чтобы их
можно было проверить вручную. Исходные записи остаются на листе `raw_records`.

In [5]:
raw_df = raw_records_df.copy(deep=True)
raw_df["cfp_id"] = raw_df["cfp_url"].map(lambda url: urlsplit(url).path.rsplit("/", 1)[-1])
scalar_columns = ["acronym", "name", "location_raw", "submission_deadline_raw", "start_date_raw"]
for column in scalar_columns:
    raw_df[column] = raw_df[column].astype("string").str.replace(r"\s+", " ", regex=True).str.strip().replace("", pd.NA)

conflict_counts = raw_df.groupby("cfp_id")[scalar_columns].nunique(dropna=True)
quality_rows = []
for cfp_id, counts in conflict_counts.iterrows():
    for field in counts.index[counts.gt(1)]:
        values = raw_df.loc[raw_df["cfp_id"].eq(cfp_id), field].dropna().unique()
        quality_rows.append({"cfp_id": cfp_id, "field": field, "issue": "conflicting_values",
                             "raw_value": " | ".join(map(str, values))})

conferences_df = raw_df.groupby(["cfp_id", "cfp_url"], as_index=False)[scalar_columns].first()
source_dates = raw_df.groupby("cfp_id")["fetched_at_utc"].agg(
    source_first_fetched_at_utc="min", source_last_fetched_at_utc="max",
)
conferences_df = conferences_df.merge(source_dates, on="cfp_id", validate="one_to_one")
conference_areas_df = (
    raw_df[["cfp_id", "area_id", "area", "area_url"]]
    .drop_duplicates(["cfp_id", "area_id"])
    .sort_values(["cfp_id", "area"], ignore_index=True)
)
conference_topics_df = (
    raw_df[["cfp_id", "topics"]].explode("topics").rename(columns={"topics": "topic"}).dropna()
)
conference_topics_df["topic"] = conference_topics_df["topic"].astype("string").str.strip()
conference_topics_df = (
    conference_topics_df.loc[conference_topics_df["topic"].ne("")]
    .drop_duplicates().sort_values(["cfp_id", "topic"], ignore_index=True)
)
area_groups = conference_areas_df.groupby("cfp_id")["area"]
conferences_df["areas"] = conferences_df["cfp_id"].map(area_groups.agg(lambda x: "; ".join(sorted(set(x)))))
conferences_df["area_count"] = conferences_df["cfp_id"].map(area_groups.nunique()).astype("Int64")
topic_groups = conference_topics_df.groupby("cfp_id")["topic"]
conferences_df["topics"] = conferences_df["cfp_id"].map(topic_groups.agg(lambda x: "; ".join(sorted(set(x)))))
print(f"Уникальных CFP: {len(conferences_df)}; принадлежностей Areas: {len(conference_areas_df)}")
display(conferences_df.head())
conflict_quality_rows = quality_rows.copy()

Уникальных CFP: 1333; принадлежностей Areas: 2438


,cfp_id,cfp_url,acronym,name,location_raw,submission_deadline_raw,start_date_raw,source_first_fetched_at_utc,source_last_fetched_at_utc,areas,area_count,topics
0,10th_ISFF2027,https://easychair.org/cfp/10th_ISFF2027,10th ISFF2027,The 10th International Symposium on Flash Floods in Wadi Systems,"Kyoto, Japan","September 30, 2026","March 29, 2027",2026-10-08T21:13:20+00:00,2026-10-08T21:13:24+00:00,Civil Engineering; Earth and Environmental Sciences,2,3d hydrological connectivity of wadis.; climate change impacts; nature based...
1,11th-IACSWS26,https://easychair.org/cfp/11th-IACSWS26,11th IACS WS'26,"11th GI/GCh ACM Workshop on Industrial Automation and Control Systems 2026, ...","Dresden, Germany","May 8, 2026","September 24, 2026",2026-10-08T21:13:22+00:00,2026-10-08T21:13:29+00:00,Computing; Engineering,2,event description languages; predictive maintenance; roles of a digital twin...
2,12_ICBF,https://easychair.org/cfp/12_ICBF,12th ICBF,The 12th International Conference on Business and Finance - “Leading through...,"Jeddah, Saudi Arabia","October 16, 2026","December 2, 2026",2026-10-08T21:13:18+00:00,2026-10-08T21:13:25+00:00,Business and Management; Economics,2,digital transformation; finance; logistics supply chain management; sustaina...
3,16thICCCNT2025,https://easychair.org/cfp/16thICCCNT2025,16th ICCCNT 2025,16th International IEEE Conference on Computing Communication and Networking...,"Indore, India","June 17, 2025","July 6, 2025",2026-10-08T21:13:22+00:00,2026-10-08T21:13:29+00:00,Computing; Engineering,2,communication 5g wireless; computing computer vision iot biometrics; data sc...
4,19thSeminarioRII,https://easychair.org/cfp/19thSeminarioRII,19th Seminario RII,XIX Seminario Internacional de Investigación sobre Globalización y Territori...,"Curitiba, Brazil","September 15, 2026","October 7, 2026",2026-10-08T21:13:25+00:00,2026-10-08T21:13:41+00:00,Economics; Social Sciences,2,geopolitics and geoeconomics; globalization and territorial transformations;...


## 5. Даты и страны для Superset

Даты превращаем в настоящие значения datetime. Пустые даты остаются пустыми;
неразобранные непустые значения сохраняем в исходных полях и списке проблем.

Страну определяем по последней части Location и справочнику ISO. Используем явные
соответствия для распространённых написаний, например UK → United Kingdom.
Если страна не распознана, оставляем её пустой и показываем запись для проверки.
Автоматического геокодирования и угадывания страны по городу нет.
Для World Map подготовлен `country_iso3`. `locality` сохраняет город/регион как текст.

In [6]:
quality_rows = conflict_quality_rows.copy()
for field in ["submission_deadline", "start_date"]:
    raw_field = f"{field}_raw"
    conferences_df[field] = pd.to_datetime(conferences_df[raw_field], format="mixed", errors="coerce")
    invalid = conferences_df[raw_field].notna() & conferences_df[field].isna()
    for row in conferences_df.loc[invalid, ["cfp_id", raw_field]].itertuples(index=False, name=None):
        quality_rows.append({"cfp_id": row[0], "field": field, "issue": "unparsed_date", "raw_value": row[1]})
conferences_df["start_year"] = conferences_df["start_date"].dt.year.astype("Int64")
conferences_df["start_month"] = conferences_df["start_date"].dt.month.astype("Int64")

COUNTRY_ALIASES = {
    "uk": "United Kingdom", "usa": "United States", "u.s.a.": "United States",
    "turkey": "TUR",
    "united states of america": "United States", "the netherlands": "Netherlands",
    "south korea": "Korea, Republic of", "republic of korea": "Korea, Republic of",
    "russia": "Russian Federation", "iran": "Iran, Islamic Republic of",
    "taiwan": "Taiwan, Province of China", "vietnam": "Viet Nam",
    "uae": "United Arab Emirates", "bolivia": "Bolivia, Plurinational State of",
    "venezuela": "Venezuela, Bolivarian Republic of", "tanzania": "Tanzania, United Republic of",
}

def location_fields(location):
    fields = {"locality": None, "country_raw": None, "country": None,
              "country_iso2": None, "country_iso3": None, "is_online": False}
    if pd.isna(location):
        return fields
    parts = [part.strip() for part in location.split(",")]
    candidate = parts[-1]
    fields["country_raw"] = candidate
    fields["is_online"] = bool(re.search(r"\b(?:online|virtual)\b", location, flags=re.IGNORECASE))
    try:
        country = pycountry.countries.lookup(COUNTRY_ALIASES.get(candidate.casefold(), candidate))
    except LookupError:
        return fields
    fields.update(country=country.name, country_iso2=country.alpha_2, country_iso3=country.alpha_3)
    if len(parts) > 1:
        fields["locality"] = ", ".join(parts[:-1])
    return fields

locations_df = pd.DataFrame(conferences_df["location_raw"].map(location_fields).tolist())
conferences_df = pd.concat([conferences_df.drop(columns=list(locations_df.columns), errors="ignore"), locations_df], axis=1)
unknown_country = conferences_df["location_raw"].notna() & conferences_df["country"].isna() & ~conferences_df["is_online"]
for cfp_id, location in conferences_df.loc[unknown_country, ["cfp_id", "location_raw"]].itertuples(index=False, name=None):
    quality_rows.append({"cfp_id": cfp_id, "field": "country", "issue": "unresolved_country", "raw_value": location})

display(conferences_df[["cfp_id", "location_raw", "country", "country_iso3", "submission_deadline", "start_date"]].head(10))

,cfp_id,location_raw,country,country_iso3,submission_deadline,start_date
0,10th_ISFF2027,"Kyoto, Japan",Japan,JPN,2026-09-30,2027-03-29
1,11th-IACSWS26,"Dresden, Germany",Germany,DEU,2026-05-08,2026-09-24
2,12_ICBF,"Jeddah, Saudi Arabia",Saudi Arabia,SAU,2026-10-16,2026-12-02
3,16thICCCNT2025,"Indore, India",India,IND,2025-06-17,2025-07-06
4,19thSeminarioRII,"Curitiba, Brazil",Brazil,BRA,2026-09-15,2026-10-07
5,1ICEIT-2027,"Taiz, Yemen",Yemen,YEM,2026-09-20,2027-02-14
6,1ST_SMPA-EU,"Vlora, Albania",Albania,ALB,2025-10-07,2025-10-18
7,2026-IISE,"Tianjin, China",China,CHN,NaT,2026-08-13
8,2026-RAPDASA-AMI-Conference,"Polokwane, South Africa",South Africa,ZAF,2026-06-08,2026-10-27
9,2026Allerton,"Urbana, Illinois, United States",United States,USA,2026-06-08,2026-09-15


## 6. Контроль качества

`area_summary_df` показывает охват каждого Area. Сумма его счётчиков может превышать
общее число CFP, потому что принадлежности разделам пересекаются.
Пропуски не заполняем придуманными значениями. Перед построением карты проверить
`unresolved_country`; правила исправления можно добавить в COUNTRY_ALIASES и повторить
ячейки подготовки, контроля и экспорта без нового скачивания.

In [7]:
quality_issues_df = pd.DataFrame(quality_rows, columns=["cfp_id", "field", "issue", "raw_value"]).drop_duplicates()
counts_by_area = conference_areas_df.groupby("area_id")["cfp_id"].nunique().rename("unique_cfps")
area_summary_df = audit_df.merge(counts_by_area, on="area_id", how="left", validate="one_to_one")
area_summary_df["unique_cfps"] = area_summary_df["unique_cfps"].fillna(0).astype("Int64")

assert conferences_df["cfp_id"].is_unique
assert conferences_df["cfp_url"].is_unique
assert not conference_areas_df.duplicated(["cfp_id", "area_id"]).any()
assert set(conference_areas_df["cfp_id"]).issubset(set(conferences_df["cfp_id"]))
assert set(conference_topics_df["cfp_id"]).issubset(set(conferences_df["cfp_id"]))
assert set(area_summary_df["area_id"]) == set(areas_df["area_id"])
assert audit_df["error"].isna().all()
has_count = audit_df["reported_count"].notna()
assert audit_df.loc[has_count, "reported_count"].eq(audit_df.loc[has_count, "parsed_count"]).all()

metadata_df = pd.DataFrame([
    ("source", AREA_INDEX_URL), ("run_at_utc", RUN_AT_UTC),
    ("index_fetched_at_utc", index_meta["fetched_at_utc"]),
    ("areas", len(areas_df)), ("raw_records", len(raw_records_df)),
    ("unique_conferences", len(conferences_df)),
    ("conference_area_memberships", len(conference_areas_df)),
    ("duplicate_same_area_rows", len(raw_df) - len(conference_areas_df)),
    ("conference_topic_memberships", len(conference_topics_df)),
    ("missing_start_date", int(conferences_df["start_date"].isna().sum())),
    ("missing_deadline", int(conferences_df["submission_deadline"].isna().sum())),
    ("missing_country", int(conferences_df["country"].isna().sum())),
    ("quality_issues", len(quality_issues_df)),
], columns=["metric", "value"])
display(metadata_df)
display(area_summary_df[["area", "reported_count", "parsed_count", "unique_cfps", "fetched_at_utc"]])
display(quality_issues_df.head(30))

,metric,value
0,source,https://easychair.org/cfp/area
1,run_at_utc,2026-10-08T21:22:12+00:00
2,index_fetched_at_utc,2026-10-08T21:13:12+00:00
3,areas,21
4,raw_records,2438
5,unique_conferences,1333
6,conference_area_memberships,2438
7,duplicate_same_area_rows,0
8,conference_topic_memberships,4002
9,missing_start_date,65


,area,reported_count,parsed_count,unique_cfps,fetched_at_utc
0,Accounting and Finance,10.0,10,10,2026-10-08T21:13:13+00:00
1,Art and Humanities,85.0,85,85,2026-10-08T21:13:15+00:00
2,Biological Sciences,24.0,24,24,2026-10-08T21:13:16+00:00
3,Business and Management,88.0,88,88,2026-10-08T21:13:18+00:00
4,Chemistry,NaN,8,8,2026-10-08T21:13:19+00:00
5,Civil Engineering,35.0,35,35,2026-10-08T21:13:20+00:00
6,Computing,609.0,609,609,2026-10-08T21:13:22+00:00
7,Earth and Environmental Sciences,43.0,43,43,2026-10-08T21:13:24+00:00
8,Economics,35.0,35,35,2026-10-08T21:13:25+00:00
9,Education Science,94.0,94,94,2026-10-08T21:13:26+00:00


,cfp_id,field,issue,raw_value


## 7. Экспорт непосредственно в XLSX

`conferences` — одна строка на CFP; основной лист для карты, дат и общего счётчика.
`conference_areas` и `conference_topics` — отдельные связи для анализа тематик.
В Superset при объединении со связями считать `COUNT(DISTINCT cfp_id)`.
`areas`/`topics` в основном листе — удобные текстовые перечни, не отдельные категории.

Остальные листы сохраняют полноту сбора, проблемы качества, исходные строки и сведения
о запуске. Даты записываются как даты Excel. XLSX не включён в .gitignore.
Текст с сайта записывается как текст, а не как формула Excel.

In [8]:
conferences_df = conferences_df[[
    "cfp_id", "acronym", "name", "country", "country_iso3", "locality",
    "submission_deadline", "start_date", "start_year", "start_month", "is_online",
    "area_count", "areas", "topics", "cfp_url", "location_raw", "country_raw",
    "submission_deadline_raw", "start_date_raw",
    "source_first_fetched_at_utc", "source_last_fetched_at_utc", "country_iso2",
]].sort_values(["start_date", "cfp_id"], na_position="last", ignore_index=True)
raw_export_df = raw_records_df.copy()
raw_export_df["topics"] = raw_export_df["topics"].map(lambda values: "; ".join(values))

sheets = {
    "conferences": conferences_df,
    "conference_areas": conference_areas_df,
    "conference_topics": conference_topics_df,
    "area_summary": area_summary_df,
    "quality_issues": quality_issues_df,
    "raw_records": raw_export_df,
    "metadata": metadata_df,
}
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
with pd.ExcelWriter(
    OUTPUT_PATH, engine="xlsxwriter", date_format="yyyy-mm-dd", datetime_format="yyyy-mm-dd",
    engine_kwargs={"options": {"strings_to_formulas": False, "strings_to_urls": False}},
) as writer:
    workbook = writer.book
    header_format = workbook.add_format({"bold": True, "bg_color": "#17365D", "font_color": "white", "text_wrap": True, "valign": "vcenter"})
    text_format = workbook.add_format({"text_wrap": True, "valign": "top"})
    date_format = workbook.add_format({"num_format": "yyyy-mm-dd", "valign": "top"})
    for sheet_name, frame in sheets.items():
        frame.to_excel(writer, sheet_name=sheet_name, index=False)
        worksheet = writer.sheets[sheet_name]
        worksheet.freeze_panes(1, 0)
        worksheet.autofilter(0, 0, len(frame), len(frame.columns) - 1)
        worksheet.set_row(0, 45)
        widths = []
        for column_index, column in enumerate(frame.columns):
            width = 60 if column in {"name", "topics", "areas", "raw_value", "value"} else 28
            if column.endswith("url"):
                width = 60
            if pd.api.types.is_datetime64_any_dtype(frame[column]):
                width = 20
            widths.append(width)
            worksheet.set_column(column_index, column_index, width, date_format if pd.api.types.is_datetime64_any_dtype(frame[column]) else text_format)
            worksheet.write(0, column_index, column, header_format)
        for row_index, row in enumerate(frame.itertuples(index=False, name=None), start=1):
            lines = max(
                max(1, len(textwrap.wrap(str(value), width=max(10, int(width) - 3))))
                for value, width in zip(row, widths)
            )
            worksheet.set_row(row_index, min(409, max(18, 15 * lines + 4)))

session.close()
print(f"Сохранено: {OUTPUT_PATH}")
print({name: len(frame) for name, frame in sheets.items()})

Сохранено: C:\Users\gorku\Documents\bmstu\m3\ИИвЗБА\ЛР-6\EasyChair-Smart-CFP\data\processed\easychair_cfp.xlsx
{'conferences': 1333, 'conference_areas': 2438, 'conference_topics': 4002, 'area_summary': 21, 'quality_issues': 0, 'raw_records': 2438, 'metadata': 13}


## 8. Проверяем сохранённый файл

Повторно читаем XLSX через pandas: число строк, уникальность CFP, связи Areas,
контрольные значения и типы дат должны сохраниться после записи.

In [9]:
saved = pd.read_excel(OUTPUT_PATH, sheet_name=None, keep_default_na=False, na_values=[""])
assert set(saved) == set(sheets)
for name, frame in sheets.items():
    assert len(saved[name]) == len(frame), name
assert saved["conferences"]["cfp_id"].is_unique
assert set(saved["conference_areas"]["cfp_id"]) == set(saved["conferences"]["cfp_id"])
has_count = saved["area_summary"]["reported_count"].notna()
assert saved["area_summary"].loc[has_count, "reported_count"].eq(saved["area_summary"].loc[has_count, "parsed_count"]).all()
for column in ["submission_deadline", "start_date"]:
    pd.testing.assert_series_equal(
        pd.to_datetime(saved["conferences"][column]), conferences_df[column],
        check_dtype=False, check_names=False,
    )
print("XLSX проверен: все листы, записи, связи и даты сохранены.")
display(saved["conferences"].head())

XLSX проверен: все листы, записи, связи и даты сохранены.


,cfp_id,acronym,name,country,country_iso3,locality,submission_deadline,start_date,start_year,start_month,is_online,area_count,areas,topics,cfp_url,location_raw,country_raw,submission_deadline_raw,start_date_raw,source_first_fetched_at_utc,source_last_fetched_at_utc,country_iso2
0,MOVE2024,MOVE2024,The Second International Measuring Ontologies for Value Enhancement Workshop,Australia,AUS,Sydney,NaT,2024-06-14,2024.0,6.0,False,2,Computing; Social Sciences,automating ontologies; enterprise architecture; environmental and social and...,https://easychair.org/cfp/MOVE2024,"Sydney, Australia",Australia,NaN,"June 14, 2024",2026-10-08T21:13:22+00:00,2026-10-08T21:13:41+00:00,AU
1,felxxviii2024,FEL XXVIII 2024,The 28th Annual Conference of the Foundation for Endangered Languages,Pakistan,PAK,Islamabad,NaT,2024-09-25,2024.0,9.0,False,2,Art and Humanities; Language and Linguistics,NaN,https://easychair.org/cfp/felxxviii2024,"Islamabad, Pakistan",Pakistan,NaN,"September 25, 2024",2026-10-08T21:13:15+00:00,2026-10-08T21:13:34+00:00,PK
2,EENVIRO2024,EENVIRO 2024,EU-CONEXUS EENVIRO Research Conference 2024,Romania,ROU,Bucharest,2024-07-31,2024-10-29,2024.0,10.0,False,2,Earth and Environmental Sciences; Energy,coastal and enviromental engineering; environmental sciences and biodiversit...,https://easychair.org/cfp/EENVIRO2024,"Bucharest, Romania",Romania,"July 31, 2024","October 29, 2024",2026-10-08T21:13:24+00:00,2026-10-08T21:13:28+00:00,RO
3,AIS_2024,AIS 2024,19th International Symposium on Applied Informatics and Related Areas,Hungary,HUN,Székesfehérvár,2024-09-23,2024-11-14,2024.0,11.0,False,2,Engineering; Technology,"education, enterprise and economics; geodesy and geoinformatics; industrial ...",https://easychair.org/cfp/AIS_2024,"Székesfehérvár, Hungary",Hungary,"September 23, 2024","November 14, 2024",2026-10-08T21:13:29+00:00,2026-10-08T21:13:43+00:00,HU
4,set-ii-2024,SET-II 2024,Second International Symposium on The Paradigm Shift: Entrepreneurship in Ne...,United States,USA,"Boston, Massachusetts",2024-11-30,2024-12-06,2024.0,12.0,False,2,Business and Management; Technology,entrepreneurial education; entrepreneurship development; innovation and tech...,https://easychair.org/cfp/set-ii-2024,"Boston, Massachusetts, United States",United States,"November 30, 2024","December 6, 2024",2026-10-08T21:13:18+00:00,2026-10-08T21:13:43+00:00,US
